In [1]:
import os, json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool # why because --> pythin-->tools

load_dotenv()

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [7]:
with open("bajaj_db.json") as f:
    db = json.load(f)


In [11]:
@tool
def get_loan_status(loan_id: str) -> dict:
    """Fetches current status of a Bajaj Finance loan from the database.
    Returns EMI amount, remaining months, outstanding balance, next due date.
    Use this when customer asks about their loan details, EMI, or balance.
    Args:
        loan_id: The loan account number (e.g., 'BFL2024001')
    """
    if loan_id not in db["loans"]:
        return {"error": f"Loan {loan_id} not found"}
    loan = db["loans"][loan_id]
    return {
        "customer_name": loan["customer_name"],
        "emi": loan["emi"],
        "remaining_months": loan["remaining_months"],
        "outstanding": loan["outstanding"],
        "next_due_date": loan["next_due_date"],
        "prepayment_charge_pct": loan["prepayment_charge_pct"]
    }


In [12]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [15]:
tools = [get_loan_status]

In [16]:
llm_with_tools = llm.bind_tools(tools)

In [18]:
result = llm_with_tools.invoke('what is the status of loan id BFL9988')

In [20]:
result.tool_calls

[{'name': 'get_loan_status',
  'args': {'loan_id': 'BFL9988'},
  'id': 'call_VWYXMmCUqYgyBkUVvkDnOYdw',
  'type': 'tool_call'}]

In [22]:
get_loan_status.invoke('BFL9988')

{'customer_name': 'Priya Mehta',
 'emi': 24500,
 'remaining_months': 204,
 'outstanding': 2180000,
 'next_due_date': '2026-05-10',
 'prepayment_charge_pct': 0.0}

In [29]:
output =llm_with_tools.invoke('what is status of loan id BFL2024001')

In [30]:
output

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 114, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_888e567758', 'id': 'chatcmpl-DYp12Pge9nP90e9keR9fs6aazZE9J', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc8e0-2cf4-7773-ad48-484a66c2a746-0', tool_calls=[{'name': 'get_loan_status', 'args': {'loan_id': 'BFL2024001'}, 'id': 'call_P3LbPkAlzEk6bEvGGHCs9bhQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 114, 'output_tokens': 21, 'total_tokens': 135, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [31]:

tool_map = {"get_loan_status": get_loan_status}
for tc in output.tool_calls:
    tool_name = tc["name"]
    tool_args = tc["args"]
    tool_id   = tc["id"]

    print(f"Executing: {tool_name}({tool_args})")

    # ─── Actually run the function ───
    result = tool_map[tool_name].invoke(tool_args)

    print(f"Result: {result}")
    print()

    # ─── Store result with its call_id ───
    # The call_id links this result back to the specific tool_call the LLM made.
    # Without it, the LLM won't know which result belongs to which call.



Executing: get_loan_status({'loan_id': 'BFL2024001'})
Result: {'customer_name': 'Rahul Sharma', 'emi': 8450, 'remaining_months': 22, 'outstanding': 185900, 'next_due_date': '2026-05-05', 'prepayment_charge_pct': 2.0}



In [32]:
system = SystemMessage(content="You are a Bajaj Finance support agent. Use tools for real data.")
user_msg = HumanMessage(content="My loan is BFL2024001. What is my current EMI?")
response = llm_with_tools.invoke([system, user_msg])


In [33]:
messages = [system, user_msg, response]

In [10]:
get_loan_status("BFL9988")

{'customer_name': 'Priya Mehta',
 'emi': 24500,
 'remaining_months': 204,
 'outstanding': 2180000,
 'next_due_date': '2026-05-10',
 'prepayment_charge_pct': 0.0}

In [3]:
def get_status(loan_id):
    """
    This function will help me to get the loan sttaus of cmy customer
    args :
    loan_id : this loan_id
    
    """
    conn = True
    if conn:
            db.execute('select * from loan_status where loan_id {}'.format(loan_id))




In [5]:
get_status.__doc__

'\nThis function will help me to get the loan sttaus of cmy customer\nargs :\nloan_id : this loan_id\n\n'